# NeurIPS 2026 replication, notebook 2: Paper B features (GPU)
Adds MAE-B/16 and ResNet-50, and the Paper B datasets. ImageNet-100 is owner-gated: build the PILOT 100-class subset as an ImageFolder (train/ and val/) on Drive and point IMAGENET100_ROOT at it, or leave None to skip. DomainNet is ~7 GB and takes ~1 h per backbone.

In [ ]:
# ---- 0. Setup: Drive, repo, packages ---------------------------------------------
# Run this cell first in every notebook. Session state is lost between sessions;
# the feature caches on Drive are not.
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess, sys, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/neurips2026_replication'); WORK.mkdir(parents=True, exist_ok=True)
REPO_ZIP = WORK / 'repo.zip'            # upload the repo zip here once (claude_replication/repo.zip), or set GIT_URL
GIT_URL = ''                            # e.g. 'https://github.com/<owner>/<repo>.git' once the repo is public
if GIT_URL:
    subprocess.run(['git','clone','--depth','1',GIT_URL,'/content/repo'], check=True)
else:
    assert REPO_ZIP.exists(), f'upload the repo zip to {REPO_ZIP}'
    subprocess.run(['rm','-rf','/content/repo','/content/repo_unz']); subprocess.run(['unzip','-q','-o',str(REPO_ZIP),'-d','/content/repo_unz'], check=True)
    inner = next(p for p in pathlib.Path('/content/repo_unz').rglob('replicate.py')).parent
    subprocess.run(['mv', str(inner), '/content/repo'], check=True)
os.chdir('/content/repo'); sys.path.insert(0, '/content/repo')
os.environ['REPLICATION_FEATURES'] = str(WORK / 'features')
os.environ['REPLICATION_ALLOW_DOWNLOADS'] = '1'   # this notebook runs on Colab only; the subprocess below inherits the download permission (pipeline/features.py::_assert_download_allowed)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44','datasets','gdown','scipy','scikit-learn'], check=True)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('features ->', os.environ['REPLICATION_FEATURES'])

In [ ]:
BACKBONES = ['dinov2_b14','clip_b16','mae_b16','resnet50','vit_in21k']
DATASETS = ['cifar100','imagenet_r','domainnet','imagenet100']
IMAGENET100_ROOT = None   # e.g. '/content/drive/MyDrive/imagenet100'

In [ ]:
# ---- 1. Determinism self-check (seconds): a fixed forward pass hashed twice per backbone ----
from pipeline.features import determinism_check
for b in BACKBONES:
    print(determinism_check(b))

In [ ]:
# ---- 2. Feature extraction (idempotent: finished caches are skipped) ----------------
# Expect roughly: CIFAR-100 6 min/backbone, ImageNet-R 4 min, CUB 2 min, OmniBenchmark 10 min on a T4.
# If the session dies, just re-run this cell; it resumes at the first missing cache.
import json, subprocess, sys
cmd = [sys.executable, 'replicate.py', 'features', '--backbones', ','.join(BACKBONES), '--datasets', ','.join(DATASETS),
       '--data-root', '/content/data', '--batch', '128']
if IMAGENET100_ROOT: cmd += ['--imagenet100-root', IMAGENET100_ROOT]
subprocess.run(cmd, check=False)
print(json.dumps(json.load(open(os.environ['REPLICATION_FEATURES'] + '/MANIFEST.json')), indent=1))